# Generate Routing Features

Generate yearly Valhalla-derived routing features from active 100 m cells and yearly POI files.

For each year, this notebook starts the matching Valhalla graph from WSL/Docker, waits for the service to become ready, writes the routing feature output, and then stops the container.

In [1]:
from __future__ import annotations

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import gc
import math
import numpy as np
import subprocess
import time

import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from tqdm.auto import tqdm


def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
FEATURE_ROOT = ROUTING_DATA / "features"
MATRIX_ROOT = ROUTING_DATA / "matrices"
STATUS_DIR = ROUTING_DATA / "status"
ROUTING_STATUS_PATH = STATUS_DIR / "routing_feature_status.csv"

VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:latest"
VALHALLA_URL = "http://localhost:8002"
VALHALLA_CPUS = 16
WSL_EXE = r"C:\Windows\System32\wsl.exe"
WSL_DISTRO = "Ubuntu"
WSL_PROJECT_ROOT = "$HOME/gruendungsanalyse"
WSL_GRAPH_ROOT = f"{WSL_PROJECT_ROOT}/data/routing/valhalla_graphs"

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
MATRIX_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\leopo\Documents\CO2_Master_Code\CO2_Masterarbeit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Run Configuration

Start with the smoke settings below. After the sample output looks right, set `MAX_ACTIVE_CELLS = None` for a full year and then expand `YEARS_TO_RUN`.

In [ ]:
# Safety switch. Set True to start Valhalla, call the API, and write output files.
RUN_ROUTING = True

# Use one year first. The notebook starts the matching yearly graph itself.
YEARS_TO_RUN = [2025]

# Products to generate in each yearly Valhalla run.
RUN_NEAREST_INFRASTRUCTURE = True
RUN_SLX_EDGE_LIST = True

# Smoke run size. Set to None for the full active-cell set.
MAX_ACTIVE_CELLS = None

# POI types currently produced by the extraction notebook.
DESTINATION_TYPES = ["rail_station", "motorway_exit", "regional_centre", "urban_centre", "higher_education"]

# Keep all POIs by default. For debugging, use a smaller number such as 20.
MAX_DESTINATIONS_PER_TYPE = None

# Valhalla matrix requests are chunked to stay below this image's matrix limit.
# Although the error says "max locations", this Valhalla endpoint behaves like a max-pairs limit:
# origins * destinations must stay <= 2500.
VALHALLA_MAX_MATRIX_PAIRS = 2500
# 20 * 100 = 2000 pairs, safely below the 2500 limit.
ORIGIN_CHUNK_SIZE = 24
DESTINATION_CHUNK_SIZE = 100

# Concurrent HTTP requests to the one yearly Valhalla server.
# Used directly for SLX and as the baseline budget for nearest-infrastructure workers.
MAX_CONCURRENT_REQUESTS = 24

# SLX edge records are flushed to parquet parts to avoid holding the whole matrix in Python RAM.
SLX_FLUSH_EDGE_RECORDS = 50_000
SLX_MAX_IN_FLIGHT_ORIGINS = MAX_CONCURRENT_REQUESTS * 4
SLX_RESUME_FROM_PARTS = True

# Main SLX matrix definition.
SLX_NETWORK_CUTOFF_M = 1000
SLX_HALF_LIFE_M = 500

# Euclidean prefilter before calling Valhalla. Must be larger than the network cutoff.
SLX_EUCLIDEAN_PREFILTER_M = 1500

# Nearest-infrastructure candidate expansion in projected Euclidean space.
# Each origin starts with the nearest 10% of destinations or everything within 10 km,
# whichever is larger, and expands in 10% steps only if nothing is routable.
NEAREST_INFRA_EUCLIDEAN_PREFILTER_M = 10_000
NEAREST_INFRA_DESTINATION_STEP_SHARE = 0.10
# Nearest-infrastructure routing runs all POI types concurrently and fans each type
# out across origin chunks. These defaults keep total request pressure near the
# previous MAX_CONCURRENT_REQUESTS setting while allowing more server-side parallelism.
NEAREST_INFRA_TYPE_WORKERS = len(DESTINATION_TYPES)
NEAREST_INFRA_ORIGIN_CHUNK_WORKERS = max(1, math.ceil(MAX_CONCURRENT_REQUESTS / max(1, NEAREST_INFRA_TYPE_WORKERS)))
NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS = NEAREST_INFRA_ORIGIN_CHUNK_WORKERS * 4

# Graph startup timeout. Increase for a slower machine or very large graph folder.
MAX_WAIT_MINUTES = 30

# Add suffix for smoke runs so they do not look like complete final outputs.
SMOKE_SUFFIX = "_sample" if MAX_ACTIVE_CELLS is not None else ""

## WSL / Docker Helpers

In [3]:
def run_local(command: list[str], check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(command, check=check, text=True, capture_output=capture_output)


def wsl_base_command() -> list[str]:
    if WSL_DISTRO:
        return [WSL_EXE, "-d", WSL_DISTRO, "--"]
    return [WSL_EXE, "--"]


def run_wsl(command: str, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return run_local([*wsl_base_command(), "bash", "-lc", command], check=check, capture_output=capture_output)


def require_wsl_distribution() -> None:
    distro_list = run_local([WSL_EXE, "-l", "-q"], check=False)
    available_distros = [line.strip().replace("\x00", "") for line in distro_list.stdout.splitlines() if line.strip().replace("\x00", "")]
    if WSL_DISTRO and available_distros and WSL_DISTRO not in available_distros:
        raise RuntimeError(
            f"Configured WSL_DISTRO={WSL_DISTRO!r}, but PowerShell reports these WSL distros: {available_distros}. "
            "Set WSL_DISTRO to the exact name from `wsl -l -v`."
        )
    test = run_wsl("printf ok", check=False)
    if test.returncode != 0:
        raise RuntimeError(
            "Could not start the configured WSL distribution. Run `wsl -l -v` in PowerShell "
            "and set WSL_DISTRO in this notebook to the exact distro name.\n\n"
            f"stdout:\n{test.stdout}\n\nstderr:\n{test.stderr}"
        )


def quote_bash(value: str) -> str:
    return "'" + value.replace("'", "'\\''") + "'"


def quote_wsl_path(value: str) -> str:
    if value.startswith("$HOME/"):
        return "$HOME/" + quote_bash(value.removeprefix("$HOME/"))
    return quote_bash(value)


def wsl_graph_dir(year: int) -> str:
    return f"{WSL_GRAPH_ROOT}/{year}"


def wsl_manifest_path(year: int) -> str:
    return f"{wsl_graph_dir(year)}/build_manifest.json"


def container_name(year: int) -> str:
    return f"co2-valhalla-routing-{year}"


def graph_manifest_exists(year: int) -> bool:
    return run_wsl(f"test -f {quote_wsl_path(wsl_manifest_path(year))}", check=False).returncode == 0


def start_valhalla_container(year: int) -> None:
    if not graph_manifest_exists(year):
        raise FileNotFoundError(f"Missing graph manifest in WSL: {wsl_manifest_path(year)}")
    name = container_name(year)
    graph_dir = wsl_graph_dir(year)
    command = " && ".join([
        f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true",
        "docker run -d "
        f"--name {quote_bash(name)} "
        f"--cpus {VALHALLA_CPUS} "
        "-p 8002:8002 "
        "-e build_admins=True "
        "-e build_time_zones=True "
        "-e build_tar=True "
        "-e serve_tiles=True "
        f"-v {quote_wsl_path(graph_dir)}:/custom_files "
        f"{quote_bash(VALHALLA_IMAGE)}",
    ])
    run_wsl(command)


def stop_valhalla_container(year: int) -> None:
    run_wsl(f"docker rm -f {quote_bash(container_name(year))} >/dev/null 2>&1 || true", check=False)


def wait_until_valhalla_ready(year: int, max_wait_minutes: int = MAX_WAIT_MINUTES) -> dict:
    deadline = time.time() + max_wait_minutes * 60
    last_error = None
    while time.time() < deadline:
        try:
            summary = valhalla_test_route()
            print(f"Valhalla ready for {year}: {summary}")
            return summary
        except Exception as error:
            last_error = error
            time.sleep(10)
    logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
    raise TimeoutError(f"Valhalla did not become ready for {year}. Last error: {last_error}\n\nContainer logs:\n{logs}")

## Routing Helpers

In [4]:
def output_paths(year: int) -> dict:
    feature_dir = FEATURE_ROOT / str(year)
    matrix_dir = MATRIX_ROOT / str(year)
    feature_dir.mkdir(parents=True, exist_ok=True)
    matrix_dir.mkdir(parents=True, exist_ok=True)
    return {
        "nearest": feature_dir / f"nearest_infrastructure_100m{SMOKE_SUFFIX}.parquet",
        "nearest_partial": feature_dir / f"nearest_infrastructure_100m{SMOKE_SUFFIX}.partial.parquet",
        "catchments": feature_dir / "accessibility_catchments_100m.parquet",
        "potentials": feature_dir / "accessibility_potentials_100m.parquet",
        "slx_edges": matrix_dir / f"W_local_drive_1km_hl500m_edges{SMOKE_SUFFIX}.parquet",
    }


def valhalla_test_route() -> dict:
    payload = {
        "locations": [
            {"lat": 47.0707, "lon": 15.4395},
            {"lat": 47.0580, "lon": 15.4630},
        ],
        "costing": "auto",
        "directions_options": {"units": "kilometers"},
    }
    response = requests.post(f"{VALHALLA_URL}/route", json=payload, timeout=30)
    response.raise_for_status()
    summary = response.json()["trip"]["summary"]
    if summary.get("time", 0) <= 0:
        raise RuntimeError(f"Valhalla responded but returned an invalid route summary: {summary}")
    return summary


def route_matrix(origins: pd.DataFrame, destinations: gpd.GeoDataFrame) -> list[list[dict]]:
    matrix_pairs = len(origins) * len(destinations)
    if matrix_pairs > VALHALLA_MAX_MATRIX_PAIRS:
        raise ValueError(
            f"Matrix request has {matrix_pairs:,} pairs "
            f"({len(origins):,} origins * {len(destinations):,} destinations), "
            f"above Valhalla limit {VALHALLA_MAX_MATRIX_PAIRS:,}. "
            "Lower ORIGIN_CHUNK_SIZE or DESTINATION_CHUNK_SIZE."
        )
    payload = {
        "sources": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in origins.itertuples(index=False)
        ],
        "targets": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in destinations.itertuples(index=False)
        ],
        "costing": "auto",
    }
    response = requests.post(f"{VALHALLA_URL}/sources_to_targets", json=payload, timeout=180)
    if not response.ok:
        raise RuntimeError(
            f"Valhalla matrix request failed with HTTP {response.status_code}: {response.text[:1000]}"
        )
    data = response.json()
    if "sources_to_targets" not in data:
        raise RuntimeError(f"Unexpected Valhalla matrix response keys: {sorted(data.keys())}")
    return data["sources_to_targets"]


def route_matrix_chunk(origins: pd.DataFrame, destinations: gpd.GeoDataFrame) -> tuple[pd.DataFrame, gpd.GeoDataFrame, list[list[dict]]]:
    return origins, destinations, route_matrix(origins, destinations)


def finite_number(value: object) -> bool:
    return isinstance(value, (int, float)) and math.isfinite(value)


def normalize_destinations(pois: gpd.GeoDataFrame, poi_type: str) -> gpd.GeoDataFrame:
    selected = pois[pois["poi_type"] == poi_type].copy()
    if selected.empty:
        return selected
    selected = selected.to_crs("EPSG:4326")
    selected["lon"] = selected.geometry.x
    selected["lat"] = selected.geometry.y
    if "poi_id" not in selected.columns:
        selected["poi_id"] = poi_type + "_" + selected.index.astype(str)
    if MAX_DESTINATIONS_PER_TYPE is not None:
        selected = selected.head(MAX_DESTINATIONS_PER_TYPE).copy()
    return selected.reset_index(drop=True)


def nearest_infrastructure_stage_size(total_destinations: int) -> int:
    return max(1, math.ceil(total_destinations * NEAREST_INFRA_DESTINATION_STEP_SHARE))


def update_best_routes(
    best_by_origin: dict,
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    matrix: list[list[dict]],
    eligible_destination_indices_by_origin: dict | None = None,
) -> None:
    for origin_row, result_row in zip(origins.itertuples(index=False), matrix):
        current_best = best_by_origin.get(origin_row.grid_id)
        eligible_indices = None
        if eligible_destination_indices_by_origin is not None:
            eligible_indices = eligible_destination_indices_by_origin.get(origin_row.grid_id)
        for destination_index, result in enumerate(result_row):
            destination_global_index = int(destinations.index[destination_index])
            if eligible_indices is not None and destination_global_index not in eligible_indices:
                continue
            if result.get("status", 0) != 0:
                continue
            result_time = result.get("time")
            result_distance = result.get("distance")
            if not finite_number(result_time) or not finite_number(result_distance):
                continue
            current_best_time = math.inf if current_best is None else current_best["result"].get("time", math.inf)
            if result_time < current_best_time:
                current_best = {
                    "result": result,
                    "destination": destinations.iloc[destination_index],
                }
        best_by_origin[origin_row.grid_id] = current_best


def best_routes_to_records(best_by_origin: dict, poi_type: str) -> pd.DataFrame:
    records = []
    for grid_id, best_payload in best_by_origin.items():
        if best_payload is None:
            records.append({
                "grid_id": grid_id,
                f"tt_{poi_type}_min": pd.NA,
                f"km_{poi_type}": pd.NA,
                f"nearest_{poi_type}_id": pd.NA,
                f"routing_status_{poi_type}": "unroutable",
            })
            continue
        best = best_payload["result"]
        destination = best_payload["destination"]
        records.append({
            "grid_id": grid_id,
            f"tt_{poi_type}_min": float(best.get("time")) / 60,
            f"km_{poi_type}": best.get("distance"),
            f"nearest_{poi_type}_id": destination.get("poi_id"),
            f"routing_status_{poi_type}": "ok",
        })
    columns = [
        "grid_id",
        f"tt_{poi_type}_min",
        f"km_{poi_type}",
        f"nearest_{poi_type}_id",
        f"routing_status_{poi_type}",
    ]
    return pd.DataFrame(records, columns=columns)


def nearest_for_origin_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    destination_x: np.ndarray,
    destination_y: np.ndarray,
    poi_type: str,
) -> tuple[pd.DataFrame, int]:
    best_by_origin = {grid_id: None for grid_id in origins["grid_id"]}
    total_destinations = len(destinations)
    stage_size = nearest_infrastructure_stage_size(total_destinations)
    radius_sq = NEAREST_INFRA_EUCLIDEAN_PREFILTER_M ** 2
    completed_requests = 0

    origin_grid_ids = origins["grid_id"].tolist()
    origin_x = origins["centroid_x_3035"].to_numpy(dtype=float)
    origin_y = origins["centroid_y_3035"].to_numpy(dtype=float)
    distance_sq = (origin_x[:, None] - destination_x[None, :]) ** 2 + (origin_y[:, None] - destination_y[None, :]) ** 2
    ordered_destination_indices = np.argsort(distance_sq, axis=1)
    initial_limits = np.maximum(stage_size, (distance_sq <= radius_sq).sum(axis=1).astype(int))
    current_limits = np.clip(initial_limits, 1, total_destinations)
    previous_limits = np.zeros(len(origins), dtype=int)
    unresolved_positions = np.arange(len(origins), dtype=int)

    while len(unresolved_positions) > 0:
        eligible_by_origin = {}
        pending_positions = []
        requested_destination_indices = set()
        for row_position in unresolved_positions:
            start = int(previous_limits[row_position])
            stop = int(current_limits[row_position])
            if start >= stop:
                continue
            candidate_indices = {
                int(index)
                for index in ordered_destination_indices[row_position, start:stop]
            }
            if not candidate_indices:
                continue
            pending_positions.append(int(row_position))
            eligible_by_origin[origin_grid_ids[row_position]] = candidate_indices
            requested_destination_indices.update(candidate_indices)

        if not requested_destination_indices:
            break

        pending_origins = origins.iloc[pending_positions].copy()
        destination_indices = sorted(requested_destination_indices)
        for destination_start in range(0, len(destination_indices), DESTINATION_CHUNK_SIZE):
            chunk_indices = destination_indices[destination_start:destination_start + DESTINATION_CHUNK_SIZE]
            destination_chunk = destinations.iloc[chunk_indices].copy()
            matrix = route_matrix(pending_origins, destination_chunk)
            update_best_routes(
                best_by_origin,
                pending_origins,
                destination_chunk,
                matrix,
                eligible_destination_indices_by_origin=eligible_by_origin,
            )
            completed_requests += 1

        previous_limits[pending_positions] = current_limits[pending_positions]
        next_unresolved_positions = [
            int(row_position)
            for row_position in unresolved_positions
            if best_by_origin[origin_grid_ids[row_position]] is None and current_limits[row_position] < total_destinations
        ]
        if not next_unresolved_positions:
            break
        current_limits[next_unresolved_positions] = np.minimum(
            total_destinations,
            current_limits[next_unresolved_positions] + stage_size,
        )
        unresolved_positions = np.array(next_unresolved_positions, dtype=int)

    return best_routes_to_records(best_by_origin, poi_type), completed_requests


def nearest_for_type(active_cells: pd.DataFrame, destinations: gpd.GeoDataFrame, poi_type: str, progress_position: int | None = None) -> pd.DataFrame:
    destinations_3035 = destinations.to_crs("EPSG:3035")
    destination_x = destinations_3035.geometry.x.to_numpy(dtype=float)
    destination_y = destinations_3035.geometry.y.to_numpy(dtype=float)
    chunk_starts = list(range(0, len(active_cells), ORIGIN_CHUNK_SIZE))
    completed_requests = 0
    chunk_results = {}
    progress_kwargs = {
        "total": len(active_cells),
        "desc": f"Nearest infrastructure: {poi_type}",
        "unit": "origin",
        "leave": True,
    }
    if progress_position is not None:
        progress_kwargs["position"] = progress_position
    progress_bar = tqdm(**progress_kwargs)

    def submit_next(executor, next_chunk_index: int, futures: dict) -> int:
        if next_chunk_index >= len(chunk_starts):
            return next_chunk_index
        chunk_start = chunk_starts[next_chunk_index]
        origins = active_cells.iloc[chunk_start:chunk_start + ORIGIN_CHUNK_SIZE].copy()
        future = executor.submit(nearest_for_origin_chunk, origins, destinations, destination_x, destination_y, poi_type)
        futures[future] = chunk_start
        return next_chunk_index + 1

    try:
        with ThreadPoolExecutor(max_workers=NEAREST_INFRA_ORIGIN_CHUNK_WORKERS) as executor:
            futures = {}
            next_chunk_index = 0
            while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                next_chunk_index = submit_next(executor, next_chunk_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    chunk_start = futures.pop(completed)
                    nearest_chunk, request_count = completed.result()
                    chunk_results[chunk_start] = nearest_chunk
                    completed_requests += request_count
                    progress_bar.update(len(nearest_chunk))
                    progress_bar.set_postfix_str(f"{completed_requests:,} matrix requests")
                    while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                        next_chunk_index = submit_next(executor, next_chunk_index, futures)
                    break
    finally:
        progress_bar.close()

    if not chunk_results:
        return best_routes_to_records({}, poi_type)
    return pd.concat([chunk_results[chunk_start] for chunk_start in chunk_starts], ignore_index=True)


def merge_nearest_results(base_output: pd.DataFrame, destination_tables: dict, nearest_results_by_type: dict) -> pd.DataFrame:
    output = base_output.copy()
    for poi_type in DESTINATION_TYPES:
        destinations = destination_tables[poi_type]
        if destinations.empty:
            output[f"tt_{poi_type}_min"] = pd.NA
            output[f"km_{poi_type}"] = pd.NA
            output[f"nearest_{poi_type}_id"] = pd.NA
            output[f"routing_status_{poi_type}"] = "missing_destinations"
            continue
        nearest = nearest_results_by_type.get(poi_type)
        if nearest is None:
            continue
        output = output.merge(nearest, on="grid_id", how="left")
    return output


def generate_nearest_infrastructure(year: int) -> Path:
    paths = output_paths(year)
    poi_path = OSM_DIR / f"austria-{year}-pois.geoparquet"
    if not poi_path.exists():
        raise FileNotFoundError(f"Missing yearly POI file: {poi_path}")

    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()

    pois = gpd.read_parquet(poi_path)
    base_output = active_cells[["grid_id"]].copy()
    available_types = sorted(pois["poi_type"].dropna().unique()) if "poi_type" in pois.columns else []
    print(f"{year}: {len(active_cells):,} origins, POI types: {available_types}")

    destination_tables = {poi_type: normalize_destinations(pois, poi_type) for poi_type in DESTINATION_TYPES}
    nearest_results_by_type = {}
    non_empty_types = []
    for poi_type in DESTINATION_TYPES:
        destinations = destination_tables[poi_type]
        if destinations.empty:
            tqdm.write(f"Skip {poi_type}: no destinations in {poi_path.name}")
            continue
        non_empty_types.append(poi_type)

    if non_empty_types:
        type_positions = {poi_type: index for index, poi_type in enumerate(non_empty_types)}
        with ThreadPoolExecutor(max_workers=min(NEAREST_INFRA_TYPE_WORKERS, len(non_empty_types))) as executor:
            futures = {
                executor.submit(
                    nearest_for_type,
                    active_cells,
                    destination_tables[poi_type],
                    poi_type,
                    type_positions[poi_type],
                ): poi_type
                for poi_type in non_empty_types
            }
            for completed in as_completed(futures):
                poi_type = futures[completed]
                nearest_results_by_type[poi_type] = completed.result()
                checkpoint = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
                checkpoint["year"] = year
                checkpoint["created_at"] = datetime.now().isoformat(timespec="seconds")
                checkpoint.to_parquet(paths["nearest_partial"], index=False)
                tqdm.write(f"Checkpointed nearest infrastructure after {poi_type} to {paths['nearest_partial']}")

    output = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
    output["year"] = year
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    nearest_tmp_path = paths["nearest"].with_name(paths["nearest"].name + ".tmp")
    output.to_parquet(nearest_tmp_path, index=False)
    nearest_tmp_path.replace(paths["nearest"])
    if paths["nearest_partial"].exists():
        paths["nearest_partial"].unlink()
    print(f"Wrote {len(output):,} rows to {paths['nearest']}")
    return paths["nearest"]


def load_active_cells_for_run() -> pd.DataFrame:
    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()
    return active_cells.reset_index(drop=True)


def active_cells_as_points(active_cells: pd.DataFrame) -> gpd.GeoDataFrame:
    required = {"grid_id", "centroid_x_3035", "centroid_y_3035", "lon", "lat"}
    missing = sorted(required - set(active_cells.columns))
    if missing:
        raise ValueError(f"Active routing cells are missing required columns: {missing}")
    return gpd.GeoDataFrame(
        active_cells.copy(),
        geometry=gpd.points_from_xy(active_cells["centroid_x_3035"], active_cells["centroid_y_3035"]),
        crs="EPSG:3035",
    )


def candidate_neighbor_indices(points_3035: gpd.GeoDataFrame, origin_index: int) -> list[int]:
    origin_geometry = points_3035.geometry.iloc[origin_index]
    query_geometry = origin_geometry.buffer(SLX_EUCLIDEAN_PREFILTER_M)
    candidate_indices = points_3035.sindex.query(query_geometry, predicate="intersects")
    origin_grid_id = points_3035["grid_id"].iloc[origin_index]
    return [int(index) for index in candidate_indices if points_3035["grid_id"].iloc[int(index)] != origin_grid_id]


def route_slx_origin(origin: pd.Series, candidates: pd.DataFrame) -> list[dict]:
    records = []
    origin_frame = pd.DataFrame([origin])
    for destination_start in range(0, len(candidates), DESTINATION_CHUNK_SIZE):
        destination_chunk = candidates.iloc[destination_start:destination_start + DESTINATION_CHUNK_SIZE].copy()
        matrix = route_matrix(origin_frame, destination_chunk)
        for destination_row, result in zip(destination_chunk.itertuples(index=False), matrix[0]):
            if result.get("status", 0) != 0:
                continue
            result_time = result.get("time")
            result_distance_km = result.get("distance")
            if not finite_number(result_time) or not finite_number(result_distance_km):
                continue
            network_distance_m = float(result_distance_km) * 1000
            if network_distance_m <= 0 or network_distance_m > SLX_NETWORK_CUTOFF_M:
                continue
            records.append({
                "origin_grid_id": origin["grid_id"],
                "destination_grid_id": destination_row.grid_id,
                "network_distance_m": network_distance_m,
                "travel_time_min": float(result_time) / 60,
                "weight_raw": math.exp(-math.log(2) * network_distance_m / SLX_HALF_LIFE_M),
            })
    return records


def route_slx_origin_by_index(active_cells: pd.DataFrame, points_3035: gpd.GeoDataFrame, origin_index: int) -> list[dict]:
    origin = active_cells.iloc[origin_index]
    candidate_indices = candidate_neighbor_indices(points_3035, origin_index)
    if not candidate_indices:
        return []
    candidates = active_cells.iloc[candidate_indices].copy().reset_index(drop=True)
    return route_slx_origin(origin, candidates)


def slx_existing_part_paths(edge_parts_dir: Path) -> list[Path]:
    return sorted(edge_parts_dir.glob("part_*.parquet"))


def completed_slx_origin_ids(part_paths: list[Path]) -> set:
    completed = set()
    for part_path in part_paths:
        part_origins = pd.read_parquet(part_path, columns=["origin_grid_id"])
        completed.update(part_origins["origin_grid_id"].dropna().unique())
    return completed


def slx_row_sums(part_paths: list[Path]) -> pd.Series:
    row_sums = pd.Series(dtype="float64")
    for part_path in part_paths:
        part = pd.read_parquet(part_path, columns=["origin_grid_id", "weight_raw"])
        part_sums = part.groupby("origin_grid_id")["weight_raw"].sum()
        row_sums = row_sums.add(part_sums, fill_value=0)
    return row_sums


def write_standardized_slx_edges(year: int, part_paths: list[Path], output_path: Path) -> int:
    columns = ["year", "origin_grid_id", "destination_grid_id", "network_distance_m", "travel_time_min", "weight_raw", "weight_rowstd"]
    if not part_paths:
        pd.DataFrame(columns=columns).to_parquet(output_path, index=False)
        return 0

    row_sums = slx_row_sums(part_paths)
    tmp_path = output_path.with_name(output_path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    total_edges = 0
    writer = None
    try:
        for part_path in part_paths:
            part = pd.read_parquet(part_path)
            part["weight_rowstd"] = part["weight_raw"] / part["origin_grid_id"].map(row_sums)
            part.insert(0, "year", year)
            part = part[columns]
            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(tmp_path, table.schema)
            writer.write_table(table)
            total_edges += len(part)
            del part, table
            gc.collect()
    finally:
        if writer is not None:
            writer.close()
    tmp_path.replace(output_path)
    return total_edges


def generate_slx_edge_list(year: int) -> Path:
    paths = output_paths(year)
    active_cells = load_active_cells_for_run()
    points_3035 = active_cells_as_points(active_cells)
    edge_parts_dir = paths["slx_edges"].with_suffix("")
    edge_parts_dir = edge_parts_dir.parent / f"{edge_parts_dir.name}_parts"
    edge_parts_dir.mkdir(parents=True, exist_ok=True)
    if not SLX_RESUME_FROM_PARTS:
        for old_part in edge_parts_dir.glob("part_*.parquet"):
            old_part.unlink()
    part_paths = slx_existing_part_paths(edge_parts_dir)
    completed_origin_ids = completed_slx_origin_ids(part_paths)
    if completed_origin_ids:
        print(f"Resuming SLX from {len(part_paths):,} part files with {len(completed_origin_ids):,} origins already represented")

    records_buffer = []
    part_number = len(part_paths)
    total_edges = sum(len(pd.read_parquet(part_path, columns=["origin_grid_id"])) for part_path in part_paths)

    def flush_records() -> None:
        nonlocal records_buffer, part_number, total_edges
        if not records_buffer:
            return
        part_number += 1
        part_path = edge_parts_dir / f"part_{part_number:05d}.parquet"
        part = pd.DataFrame(records_buffer)
        total_edges += len(part)
        part.to_parquet(part_path, index=False)
        part_paths.append(part_path)
        records_buffer = []
        gc.collect()

    def submit_next(executor, next_origin_index: int, futures: dict) -> int:
        while next_origin_index < len(active_cells):
            grid_id = active_cells["grid_id"].iloc[next_origin_index]
            if grid_id not in completed_origin_ids:
                future = executor.submit(route_slx_origin_by_index, active_cells, points_3035, next_origin_index)
                futures[future] = next_origin_index
                return next_origin_index + 1
            next_origin_index += 1
        return next_origin_index

    progress_bar = tqdm(total=len(active_cells), desc="SLX local edge list", unit="origin", initial=len(completed_origin_ids), leave=True)
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:
        futures = {}
        next_origin_index = 0
        while next_origin_index < len(active_cells) and len(futures) < SLX_MAX_IN_FLIGHT_ORIGINS:
            next_origin_index = submit_next(executor, next_origin_index, futures)

        completed_origins = 0
        while futures:
            for completed in as_completed(list(futures)):
                futures.pop(completed)
                origin_records = completed.result()
                if origin_records:
                    records_buffer.extend(origin_records)
                    if len(records_buffer) >= SLX_FLUSH_EDGE_RECORDS:
                        flush_records()
                completed_origins += 1
                progress_bar.update(1)
                progress_bar.set_postfix_str(f"{total_edges + len(records_buffer):,} edges")
                while next_origin_index < len(active_cells) and len(futures) < SLX_MAX_IN_FLIGHT_ORIGINS:
                    next_origin_index = submit_next(executor, next_origin_index, futures)
                break
    progress_bar.close()
    flush_records()

    total_edges = write_standardized_slx_edges(year, part_paths, paths["slx_edges"])
    print(f"Wrote {total_edges:,} SLX edges to {paths['slx_edges']}")
    return paths["slx_edges"]

## Preflight

This confirms WSL and Docker are reachable from the notebook kernel.

In [5]:
require_wsl_distribution()
whoami = run_wsl("whoami").stdout.strip()
docker_version = run_wsl("docker --version").stdout.strip()
print(f"WSL user: {whoami}")
print(docker_version)
print(f"Graph root: {WSL_GRAPH_ROOT}")

WSL user: leo
Docker version 29.5.3, build d1c06ef
Graph root: $HOME/gruendungsanalyse/data/routing/valhalla_graphs


## Run Nearest Infrastructure

With `RUN_ROUTING = False`, this cell only prints what it would do. With `RUN_ROUTING = True`, it starts the yearly Valhalla container, runs the feature product, and stops the container for each year.

In [6]:
for year in YEARS_TO_RUN:
    paths = output_paths(year)
    poi_path = OSM_DIR / f"austria-{year}-pois.geoparquet"
    active_count = len(pd.read_parquet(ACTIVE_CELLS_PATH, columns=["grid_id"]))
    print(f"Year {year}")
    print(f"  active cells: {active_count:,}")
    print(f"  POI file exists: {poi_path.exists()} ({poi_path})")
    print(f"  graph manifest exists in WSL: {graph_manifest_exists(year)} ({wsl_manifest_path(year)})")
    print(f"  nearest output: {paths['nearest']}")
    print(f"  SLX edge output: {paths['slx_edges']}")

    if not RUN_ROUTING:
        print("  dry run only; set RUN_ROUTING = True to start Valhalla, route, and write output")
        continue

    status_rows = []
    started_at = datetime.now().isoformat(timespec="seconds")
    try:
        start_valhalla_container(year)
        wait_until_valhalla_ready(year)
        if RUN_NEAREST_INFRASTRUCTURE:
            product_started_at = datetime.now().isoformat(timespec="seconds")
            output_path = generate_nearest_infrastructure(year)
            status_rows.append({
                "year": year,
                "product": "nearest_infrastructure_100m" + SMOKE_SUFFIX,
                "status": "done",
                "started_at": product_started_at,
                "finished_at": datetime.now().isoformat(timespec="seconds"),
                "output_path": str(output_path),
                "error_message": "",
            })
        if RUN_SLX_EDGE_LIST:
            product_started_at = datetime.now().isoformat(timespec="seconds")
            output_path = generate_slx_edge_list(year)
            status_rows.append({
                "year": year,
                "product": "W_local_drive_1km_hl500m_edges" + SMOKE_SUFFIX,
                "status": "done",
                "started_at": product_started_at,
                "finished_at": datetime.now().isoformat(timespec="seconds"),
                "output_path": str(output_path),
                "error_message": "",
            })
    except Exception as error:
        status_rows.append({
            "year": year,
            "product": "routing_generation" + SMOKE_SUFFIX,
            "status": "failed",
            "started_at": started_at,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "output_path": "",
            "error_message": repr(error),
        })
        raise
    finally:
        stop_valhalla_container(year)
        if status_rows:
            status_row = pd.DataFrame(status_rows)
            if ROUTING_STATUS_PATH.exists():
                previous_status = pd.read_csv(ROUTING_STATUS_PATH)
                status = pd.concat([previous_status, status_row], ignore_index=True)
            else:
                status = status_row
            status.to_csv(ROUTING_STATUS_PATH, index=False)

Year 2025
  active cells: 125,473
  POI file exists: True (C:\Users\leopo\Documents\CO2_Master_Code\CO2_Masterarbeit\TOOLS\osm-data\austria-2025-pois.geoparquet)
  graph manifest exists in WSL: True ($HOME/gruendungsanalyse/data/routing/valhalla_graphs/2025/build_manifest.json)
  nearest output: C:\Users\leopo\Documents\CO2_Master_Code\CO2_Masterarbeit\ANAL\data\routing\features\2025\nearest_infrastructure_100m_sample.parquet
  SLX edge output: C:\Users\leopo\Documents\CO2_Master_Code\CO2_Masterarbeit\ANAL\data\routing\matrices\2025\W_local_drive_1km_hl500m_edges_sample.parquet
Valhalla ready for 2025: {'has_time_restrictions': False, 'has_toll': False, 'has_highway': False, 'has_ferry': False, 'min_lat': 47.058115, 'min_lon': 15.43394, 'max_lat': 47.071534, 'max_lon': 15.464347, 'time': 914.682, 'length': 4.06, 'cost': 1959.05}
2025: 250 origins, POI types: ['motorway_exit', 'rail_station']


Nearest infrastructure:   6%|▋         | 13/208 [00:13<02:21,  1.37matrix/s, rail_station, 13/26 chunk requests]

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))